# Amodal completion — first baseline experiment

**Goal:** generate one complete-object image on a free T4 and record resource use.
A synthetic smoke test completed on a free Tesla T4; it verifies inference runs, not real-photo quality. The notebook installs an
isolated Python environment so the original research code does not overwrite
Colab's own PyTorch installation. No API key, billing account, or paid service is used.

Select **Runtime → Change runtime type → T4 GPU**. Run cells in order.
The official checkpoint is approximately 15.5 GB; all model downloads happen on Colab.


In [ ]:
import sys
import torch
print('Python:', sys.version)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Please select the free T4 GPU runtime.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))


## 1. Prepare the isolated environment
This cell writes the helper files and installs a separate Python 3.10 environment.
The upstream repositories' exact revisions are saved to `environment_manifest.json`.
Do not proceed if setup exits with an error.


In [ ]:
from pathlib import Path
import subprocess
import sys
ROOT = Path('/content/amodal-completion')
ROOT.mkdir(exist_ok=True)
FILES = {'requirements-baseline.txt': "# Isolated Python 3.10 environment; do not install into Colab's base interpreter.\nnumpy==1.26.4\ntorch==2.1.2\ntorchvision==0.16.2\npytorch-lightning==1.9.5\ntorchmetrics==0.11.4\nomegaconf==2.3.0\ntransformers==4.30.2\neinops==0.6.1\nkornia==0.6.12\nPillow==10.4.0\nopencv-python-headless==4.10.0.84\nsegment-anything==1.0\nftfy==6.2.3\nregex\ntqdm\nrequests\nscipy\ntensorboard\nmatplotlib\n", 'scripts/setup_colab.py': '"""Create an isolated baseline environment in a Colab Linux runtime."""\nfrom pathlib import Path\nimport json\nimport os\nimport subprocess\nimport sys\n\nROOT = Path(\'/content/amodal-completion\')\nROOT.mkdir(exist_ok=True)\n\ndef run(args, **kwargs):\n    subprocess.run([str(x) for x in args], check=True, **kwargs)\n\nrun([sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'uv\'])\nuv = [sys.executable, \'-m\', \'uv\']\ninstall_env = os.environ.copy()\n# Colab\'s CPU and CUDA package indexes expose different wheels. uv\'s default\n# first-index policy can incorrectly conclude torchmetrics is missing from the\n# CUDA index even when it is available on PyPI.\ninstall_env[\'UV_INDEX_STRATEGY\'] = \'unsafe-best-match\'\nenv = ROOT / \'.venv\'\nrun(uv + [\'venv\', \'--python\', \'3.10\', str(env)])\npython = env / \'bin/python\'\nrun(uv + [\'pip\', \'install\', \'--python\', str(python), \'-r\',\n          str(ROOT/\'requirements-baseline.txt\'),\n          \'--extra-index-url\', \'https://download.pytorch.org/whl/cu118\'],\n    env=install_env)\n# PyTorch Lightning 1.9 still imports pkg_resources, which setuptools removed\n# from recent releases. Pin the compatibility version explicitly.\nrun(uv + [\'pip\', \'install\', \'--python\', str(python), \'setuptools<81\',\n          \'--index-url\', \'https://pypi.org/simple\'])\n\nthird_party = ROOT / \'third_party\'\nthird_party.mkdir(exist_ok=True)\nrepositories = {\n    \'pix2gestalt\': \'https://github.com/cvlab-columbia/pix2gestalt.git\',\n    \'taming-transformers\': \'https://github.com/CompVis/taming-transformers.git\',\n    \'CLIP\': \'https://github.com/openai/CLIP.git\',\n}\nrevisions = {}\nfor name, url in repositories.items():\n    dest = third_party/name\n    if not dest.exists():\n        run([\'git\', \'clone\', \'--depth\', \'1\', url, str(dest)])\n    revisions[name] = subprocess.check_output(\n        [\'git\', \'-C\', str(dest), \'rev-parse\', \'HEAD\'], text=True).strip()\n    if name != \'pix2gestalt\':\n        run(uv + [\'pip\',\'install\',\'--python\',str(python),\'--no-deps\',\'-e\',str(dest)])\n\n# The old Lightning import moved; make the compatibility edit explicit and recorded.\nddpm = third_party/\'pix2gestalt/pix2gestalt/ldm/models/diffusion/ddpm.py\'\nsource = ddpm.read_text()\nold = \'from pytorch_lightning.utilities.distributed import rank_zero_only\'\nnew = \'from pytorch_lightning.utilities.rank_zero import rank_zero_only\'\nddpm.write_text(source.replace(old, new))\nmanifest = {\'repositories\': revisions, \'patches\': [f\'{old} -> {new}\'],\n            \'python\': str(python)}\n(ROOT/\'environment_manifest.json\').write_text(json.dumps(manifest, indent=2))\ncheck = subprocess.run([str(python), \'-c\',\n    "import torch, pytorch_lightning; print(\'Isolated PyTorch:\', torch.__version__); print(\'Lightning:\', pytorch_lightning.__version__); print(\'CUDA:\', torch.cuda.is_available()); assert torch.cuda.is_available(), \'Select a T4 GPU runtime before continuing.\'"],\n    check=True, text=True, capture_output=True)\nprint(check.stdout, end=\'\')\nprint(\'Environment created. Full model inference is the next separate check.\')\n', 'scripts/run_baseline.py': '"""Run one or more Pix2Gestalt RGB completion candidates in isolated Colab."""\nfrom pathlib import Path\nfrom contextlib import nullcontext\nimport argparse\nimport gc\nimport json\nimport os\nimport sys\nimport time\n\ndef prepare_inputs(image_path, mask_path=None):\n    import numpy as np\n    from PIL import Image\n    image = Image.open(image_path).convert(\'RGB\')\n    if mask_path is None:\n        raise ValueError(\'Pix2Gestalt requires a visible-object mask. For obstruction-only marking, use notebooks/02_marked_obstruction_colab.ipynb.\')\n    mask = Image.open(mask_path).convert(\'L\')\n    if image.size != mask.size:\n        raise ValueError(\'Image and visible mask must have identical dimensions.\')\n    binary = np.array(mask) >= 128\n    if not binary.any() or binary.all():\n        raise ValueError(\'Visible mask must include target pixels and background pixels.\')\n    image = image.resize((256,256), Image.Resampling.LANCZOS)\n    mask = Image.fromarray(binary.astype(\'uint8\')*255).resize((256,256), Image.Resampling.NEAREST)\n    return np.array(image), np.repeat(np.array(mask)[...,None],3,axis=2)\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--root\',type=Path,default=Path(\'/content/amodal-completion\'))\n    p.add_argument(\'--image\',type=Path,required=True)\n    p.add_argument(\'--mask\',type=Path,required=True,\n                   help=\'Binary mask of the visible target object (white=visible target).\')\n    p.add_argument(\'--scene\',type=Path,\n                   help=\'Optional untouched full scene. Provide this with --crop to enable scene composition.\')\n    p.add_argument(\'--crop\',type=int,nargs=4,metavar=(\'X0\',\'Y0\',\'X1\',\'Y1\'),\n                   help=\'Crop coordinates in --scene used to create --image.\')\n    p.add_argument(\'--occluder-mask\',type=Path,\n                   help=\'Optional obstruction mask saved with the run for full-scene composition.\')\n    p.add_argument(\'--checkpoint\',type=Path,required=True)\n    p.add_argument(\'--steps\',type=int,default=50)\n    p.add_argument(\'--seed\',type=int,default=42)\n    p.add_argument(\'--candidates\',type=int,default=1,\n                   help=\'Number of stochastic completions to generate (1-4).\')\n    args=p.parse_args()\n    if args.steps < 1 or args.steps > 1000:\n        p.error(\'steps must be between 1 and 1000\')\n    if args.candidates < 1 or args.candidates > 4:\n        p.error(\'candidates must be between 1 and 4\')\n    if bool(args.scene) != bool(args.crop):\n        p.error(\'--scene and --crop must be supplied together\')\n    if args.occluder_mask and not args.scene:\n        p.error(\'--occluder-mask requires --scene and --crop\')\n    args.image=args.image.resolve()\n    args.mask=args.mask.resolve()\n    args.checkpoint=args.checkpoint.resolve()\n    args.root=args.root.resolve()\n    if args.scene:\n        args.scene=args.scene.resolve()\n    if args.occluder_mask:\n        args.occluder_mask=args.occluder_mask.resolve()\n    scene_metadata=None\n    if args.scene:\n        from PIL import Image\n        with Image.open(args.scene) as scene_image:\n            scene_width,scene_height=scene_image.size\n            x0,y0,x1,y1=args.crop\n            if not (0 <= x0 < x1 <= scene_width and 0 <= y0 < y1 <= scene_height):\n                p.error(f\'--crop {args.crop} is outside scene dimensions {(scene_width,scene_height)}\')\n            crop_aspect=(x1-x0)/(y1-y0)\n        with Image.open(args.image) as model_input:\n            input_aspect=model_input.width/model_input.height\n        if abs(crop_aspect-input_aspect) > 1e-3:\n            p.error(\'The scene crop aspect ratio must match --image; padding/rotation transforms are not supported.\')\n        scene_metadata={\'crop_xyxy\':args.crop,\'scene_size\':[scene_width,scene_height],\n                        \'source_scene\':str(args.scene),\'input_crop_size\':[x1-x0,y1-y0],\n                        \'model_input_resolution\':[256,256]}\n    image, mask = prepare_inputs(args.image,args.mask)\n    upstream = args.root/\'third_party/pix2gestalt/pix2gestalt\'\n    sys.path.insert(0,str(upstream))\n    os.chdir(upstream)\n    import numpy as np\n    import torch\n    from PIL import Image\n    from omegaconf import OmegaConf\n    from ldm.util import instantiate_from_config\n    from inference import run_pix2gestalt\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Select a T4 GPU runtime before running this experiment.\')\n    torch.manual_seed(args.seed)\n    np.random.seed(args.seed)\n    torch.cuda.reset_peak_memory_stats()\n    config = OmegaConf.load(upstream/\'configs/sd-finetune-pix2gestalt-c_concat-256.yaml\')\n    config.model.params.use_ema = False\n    # The published checkpoint includes training state. Map it from disk instead of\n    # materializing all optimizer tensors inside the limited Colab host RAM.\n    checkpoint = torch.load(str(args.checkpoint),map_location=\'cpu\',mmap=True,weights_only=False)\n    state = checkpoint[\'state_dict\']\n    with torch.device(\'cpu\'):\n        model=instantiate_from_config(config.model)\n    selected={}\n    ema_count=0\n    for name,tensor in model.state_dict().items():\n        key=\'model_ema.\'+name[len(\'model.\'):].replace(\'.\',\'\') if name.startswith(\'model.\') else \'\'\n        if key and key in state:\n            selected[name]=state[key]\n            ema_count+=1\n        elif name in state:\n            selected[name]=state[name]\n    missing,unexpected=model.load_state_dict(selected,strict=False)\n    if missing:\n        raise RuntimeError(\'Checkpoint is missing model tensors: \'+str(missing[:15]))\n    if not ema_count:\n        raise RuntimeError(\'EMA weights were not found; stop rather than silently change the baseline.\')\n    del selected,state,checkpoint\n    gc.collect()\n    model.eval().requires_grad_(False).to(\'cuda\')\n    torch.cuda.synchronize()\n    start=time.perf_counter()\n    with torch.inference_mode():\n        results=run_pix2gestalt(model,\'cuda\',image,mask,scale=2.0,n_samples=args.candidates,\n                               ddim_steps=args.steps,precision=\'autocast\')\n    torch.cuda.synchronize()\n    elapsed=time.perf_counter()-start\n    run=args.root/\'runs\'/time.strftime(\'%Y%m%d-%H%M%S\')\n    run.mkdir(parents=True,exist_ok=True)\n    Image.fromarray(image).save(run/\'input.png\')\n    Image.fromarray(mask[:,:,0]).save(run/\'visible_mask.png\')\n    if args.scene:\n        Image.open(args.scene).convert(\'RGB\').save(run/\'scene.png\')\n        (run/\'scene_metadata.json\').write_text(json.dumps(scene_metadata,indent=2)+\'\\n\')\n    if args.occluder_mask:\n        Image.open(args.occluder_mask).convert(\'L\').save(run/\'occluder_mask.png\')\n    # Save every sample for side-by-side human selection. Keep completion.png as\n    # candidate 1 so existing tools and archives remain compatible. Pix2Gestalt\n    # returns an isolated RGB crop; this runner does not composite into the scene.\n    for index, result in enumerate(results, start=1):\n        Image.fromarray(result).save(run/f\'candidate_{index:02d}.png\')\n    Image.fromarray(results[0]).save(run/\'completion.png\')\n    report={\'seed\':args.seed,\'steps\':args.steps,\'guidance_scale\':2.0,\'candidates\':len(results),\n            \'resolution\':[256,256],\'gpu\':torch.cuda.get_device_name(),\n            \'inference_seconds\':elapsed,\n            \'peak_allocated_gib\':torch.cuda.max_memory_allocated()/2**30,\n            \'ema_tensors_loaded\':ema_count,\'torch\':torch.__version__,\n            \'scene_metadata\':scene_metadata,\n            \'occluder_mask_saved\':bool(args.occluder_mask),\n            \'status\':\'inference_completed\',\'mask_export\':\'not implemented\',\n            \'status\':\'inference_completed\'}\n    env=args.root/\'environment_manifest.json\'\n    if env.exists(): report[\'environment\']=json.loads(env.read_text())\n    (run/\'metrics.json\').write_text(json.dumps(report,indent=2))\n    print(json.dumps(report,indent=2))\n    print(\'Saved run:\',run)\n\nif __name__==\'__main__\':\n    main()\n', 'scripts/evaluate_rgb_baseline_batch.py': '"""Run the Pix2Gestalt RGB completion baseline on a fixed evaluation archive.\n\nThis script is intended for the isolated Linux environment in the Colab notebook.\nIt runs one sample at a time through run_baseline.py, saves every completion and\nerror, and creates a contact sheet for qualitative review. Ground-truth masks in\nthe archive are never passed to the generator.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nfrom PIL import Image, ImageDraw, ImageFont\n\n\ndef safe_member(archive: zipfile.ZipFile, name: str) -> bytes:\n    """Read a named regular archive member without extracting arbitrary paths."""\n    path = PurePosixPath(name)\n    if path.is_absolute() or ".." in path.parts:\n        raise ValueError(f"Unsafe archive path in manifest: {name}")\n    return archive.read(name)\n\n\ndef overlay_visible(image: Image.Image, mask: Image.Image) -> Image.Image:\n    base = image.convert("RGB").resize((256, 256), Image.Resampling.LANCZOS)\n    alpha = mask.convert("L").resize((256, 256), Image.Resampling.NEAREST)\n    tint = Image.new("RGBA", base.size, (255, 178, 35, 0))\n    tint.putalpha(alpha.point(lambda value: round(value * 0.45)))\n    return Image.alpha_composite(base.convert("RGBA"), tint).convert("RGB")\n\n\ndef make_contact_sheet(records: list[dict], path: Path) -> None:\n    tile_w, tile_h, label_h = 256, 256, 38\n    gap, header_h = 12, 34\n    rows = max(1, len(records))\n    sheet = Image.new("RGB", (2 * tile_w + 3 * gap,\n                               header_h + rows * (tile_h + label_h + gap) + gap),\n                      (31, 36, 44))\n    draw = ImageDraw.Draw(sheet)\n    font = ImageFont.load_default()\n    draw.text((gap, 10), "INPUT + VISIBLE REGION", fill="white", font=font)\n    draw.text((2 * gap + tile_w, 10), "PIX2GESTALT RGB COMPLETION", fill="white", font=font)\n    for row, record in enumerate(records):\n        y = header_h + gap + row * (tile_h + label_h + gap)\n        for col, key in enumerate(("review_input", "completion")):\n            x = gap + col * (tile_w + gap)\n            if record.get(key) and Path(record[key]).is_file():\n                tile = Image.open(record[key]).convert("RGB").resize(\n                    (tile_w, tile_h), Image.Resampling.LANCZOS)\n                sheet.paste(tile, (x, y))\n            else:\n                draw.rectangle((x, y, x + tile_w, y + tile_h), fill=(65, 70, 80))\n                draw.text((x + 8, y + 12), "Inference failed", fill="white", font=font)\n        note = (f"{record[\'sample_id\']}  |  {record.get(\'category\', \'\')}  |  "\n                f"hidden {float(record.get(\'occluded_fraction\', 0)):.0%}  |  {record[\'status\']}")\n        draw.text((gap, y + tile_h + 7), note[:115], fill=(220, 225, 235), font=font)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    sheet.save(path)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--root", type=Path, default=Path("/content/amodal-completion"))\n    parser.add_argument("--archive", type=Path, required=True)\n    parser.add_argument("--checkpoint", type=Path, required=True)\n    parser.add_argument("--steps", type=int, default=50)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    if not 1 <= args.steps <= 1000:\n        parser.error("--steps must be between 1 and 1000")\n    if not args.archive.is_file() or not args.checkpoint.is_file():\n        parser.error("The evaluation archive and model checkpoint must both exist")\n\n    run_script = args.root / "scripts/run_baseline.py"\n    out_root = args.root / "runs/rgb-eval-v1"\n    out_root.mkdir(parents=True, exist_ok=True)\n    input_root = args.root / "inputs/rgb-eval-v1"\n    input_root.mkdir(parents=True, exist_ok=True)\n\n    with zipfile.ZipFile(args.archive) as archive:\n        manifest = [json.loads(line) for line in safe_member(archive, "manifest.jsonl").decode().splitlines()\n                    if line.strip()]\n        records: list[dict] = []\n        for entry in manifest:\n            sample_id = str(entry["sample_id"])\n            sample_in = input_root / sample_id\n            sample_out = out_root / sample_id\n            sample_in.mkdir(parents=True, exist_ok=True)\n            sample_out.mkdir(parents=True, exist_ok=True)\n\n            # Read only the RGB input and the visible-target mask. Ground-truth\n            # complete/occluder masks remain untouched and are never model inputs.\n            image_bytes = safe_member(archive, entry["image"])\n            mask_bytes = safe_member(archive, entry["visible_mask"])\n            (sample_in / "image.png").write_bytes(image_bytes)\n            (sample_in / "visible_mask.png").write_bytes(mask_bytes)\n            image = Image.open(sample_in / "image.png").convert("RGB")\n            mask = Image.open(sample_in / "visible_mask.png").convert("L")\n            review_input = sample_out / "input_visible_overlay.png"\n            overlay_visible(image, mask).save(review_input)\n            shutil.copy2(sample_in / "image.png", sample_out / "input.png")\n            shutil.copy2(sample_in / "visible_mask.png", sample_out / "visible_mask.png")\n\n            command = [sys.executable, str(run_script), "--root", str(args.root),\n                       "--image", str(sample_in / "image.png"),\n                       "--mask", str(sample_in / "visible_mask.png"),\n                       "--checkpoint", str(args.checkpoint),\n                       "--steps", str(args.steps), "--seed", str(args.seed)]\n            started = time.perf_counter()\n            completed = subprocess.run(command, text=True, capture_output=True, check=False)\n            elapsed = time.perf_counter() - started\n            record = {\n                "sample_id": sample_id,\n                "category": entry.get("category", ""),\n                "occluded_fraction": entry.get("occluded_fraction", ""),\n                "status": "completed" if completed.returncode == 0 else "failed",\n                "wall_seconds": round(elapsed, 2),\n                "review_input": str(review_input),\n                "completion": "",\n            }\n            report = {\n                "sample_id": sample_id,\n                "category": record["category"],\n                "occluded_fraction": record["occluded_fraction"],\n                "status": record["status"],\n                "wall_seconds": record["wall_seconds"],\n                "settings": {"resolution": 256, "steps": args.steps,\n                             "guidance_scale": 2.0, "seed": args.seed,\n                             "candidates": 1},\n                "stdout": completed.stdout[-12000:],\n                "stderr": completed.stderr[-12000:],\n            }\n            if completed.returncode == 0:\n                match = re.search(r"Saved run:\\s*(.+)", completed.stdout)\n                if match:\n                    generated = Path(match.group(1).strip()) / "completion.png"\n                    if generated.is_file():\n                        output = sample_out / "completion.png"\n                        shutil.copy2(generated, output)\n                        record["completion"] = str(output)\n                        report_path = generated.parent / "metrics.json"\n                        if report_path.is_file():\n                            report["model_metrics"] = json.loads(report_path.read_text())\n                    else:\n                        record["status"] = report["status"] = "failed"\n                        report["stderr"] += "\\nRunner returned success but completion.png was missing."\n                else:\n                    record["status"] = report["status"] = "failed"\n                    report["stderr"] += "\\nCould not locate the run output path in runner output."\n            (sample_out / "metrics.json").write_text(json.dumps(report, indent=2))\n            if record["status"] == "failed":\n                (sample_out / "error.txt").write_text(report["stderr"] or report["stdout"])\n            records.append(record)\n            print(f"{record[\'status\']}: {sample_id} ({record[\'wall_seconds\']} s)", flush=True)\n\n    fields = ["sample_id", "category", "occluded_fraction", "status", "wall_seconds",\n              "review_input", "completion", "visible_preservation", "object_plausibility",\n              "completion_plausibility", "boundary_artifacts", "review_notes"]\n    with (out_root / "results.csv").open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields)\n        writer.writeheader()\n        for record in records:\n            writer.writerow({key: record.get(key, "") for key in fields})\n    make_contact_sheet(records, out_root / "contact_sheet.png")\n    summary = {\n        "archive": args.archive.name,\n        "samples": len(records),\n        "completed": sum(item["status"] == "completed" for item in records),\n        "failed": sum(item["status"] != "completed" for item in records),\n        "settings": {"resolution": 256, "steps": args.steps,\n                     "guidance_scale": 2.0, "seed": args.seed, "candidates": 1},\n        "quality_assessment": "RGB completions require qualitative review; no pixelwise RGB score is claimed.",\n    }\n    (out_root / "summary.json").write_text(json.dumps(summary, indent=2))\n    archive_path = shutil.make_archive(str(args.root / "runs/rgb-eval-v1"), "zip",\n                                       root_dir=out_root)\n    print(json.dumps(summary, indent=2))\n    print("Contact sheet:", out_root / "contact_sheet.png")\n    print("Results archive:", archive_path)\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/compose_rgb_results.py': '"""Make experimental scene previews from isolated Pix2Gestalt RGB outputs.\n\nThe generated crop is segmented with GrabCut, seeded by border-connected near-white\nbackground and the supplied visible-object mask. Small disconnected regions are\nremoved. Source pixels inside the visible-object mask are preserved. This remains a\nvisualization aid: generated foreground masks and scene occlusion order are unverified.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import deque\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\n\n\ndef border_background(rgb: np.ndarray, white_threshold: int) -> np.ndarray:\n    """Return near-white pixels connected to the image border."""\n    candidate = np.all(rgb >= 255 - white_threshold, axis=2)\n    height, width = candidate.shape\n    background = np.zeros_like(candidate)\n    queue: deque[tuple[int, int]] = deque()\n\n    for x in range(width):\n        if candidate[0, x]:\n            background[0, x] = True\n            queue.append((0, x))\n        if candidate[height - 1, x] and not background[height - 1, x]:\n            background[height - 1, x] = True\n            queue.append((height - 1, x))\n    for y in range(height):\n        if candidate[y, 0] and not background[y, 0]:\n            background[y, 0] = True\n            queue.append((y, 0))\n        if candidate[y, width - 1] and not background[y, width - 1]:\n            background[y, width - 1] = True\n            queue.append((y, width - 1))\n\n    while queue:\n        y, x = queue.popleft()\n        for ny, nx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):\n            if 0 <= ny < height and 0 <= nx < width and candidate[ny, nx] and not background[ny, nx]:\n                background[ny, nx] = True\n                queue.append((ny, nx))\n    return background\n\n\ndef estimate_foreground(rgb: np.ndarray, visible: np.ndarray, white_threshold: int) -> np.ndarray:\n    """Use GrabCut with image-edge background and visible-object foreground seeds."""\n    # Keep OpenCV optional for callers that provide an explicit completion mask.\n    import cv2\n\n    border_bg = border_background(rgb, white_threshold)\n    near_white = np.all(rgb >= 255 - max(white_threshold, 24), axis=2)\n    labels = np.full(near_white.shape, cv2.GC_PR_BGD, dtype=np.uint8)\n    labels[~near_white] = cv2.GC_PR_FGD\n    labels[border_bg] = cv2.GC_BGD\n\n    sure_fg = visible & ~border_bg\n    if int(sure_fg.sum()) < 8:\n        candidates = (~border_bg).astype(np.uint8)\n        count, components, stats, _ = cv2.connectedComponentsWithStats(candidates, 8)\n        if count > 1:\n            largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))\n            sure_fg = components == largest\n    labels[sure_fg] = cv2.GC_FGD\n\n    if not np.any(labels == cv2.GC_BGD) or not np.any(labels == cv2.GC_FGD):\n        return ~border_bg\n    bg_model = np.zeros((1, 65), dtype=np.float64)\n    fg_model = np.zeros((1, 65), dtype=np.float64)\n    try:\n        cv2.grabCut(rgb.copy(), labels, None, bg_model, fg_model, 4, cv2.GC_INIT_WITH_MASK)\n    except cv2.error:\n        return ~border_bg\n\n    foreground = (labels == cv2.GC_FGD) | (labels == cv2.GC_PR_FGD)\n    count, components, stats, _ = cv2.connectedComponentsWithStats(foreground.astype(np.uint8), 8)\n    if count <= 1:\n        return foreground\n    min_area = max(32, int(foreground.size * 0.002))\n    keep = np.zeros(count, dtype=bool)\n    keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= min_area\n    if not keep.any():\n        keep[1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))] = True\n    return keep[components]\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--results", type=Path, default=Path("runs/eval-v1"))\n    parser.add_argument("--output", type=Path, default=Path("runs/eval-v1/composites-v1"))\n    parser.add_argument("--white-threshold", type=int, default=12,\n                        help="Allow border-background pixels with each channel >= 255 - threshold.")\n    args = parser.parse_args()\n    if not 0 <= args.white_threshold <= 254:\n        parser.error("--white-threshold must be in [0, 254]")\n    results = args.results.resolve()\n    output = args.output.resolve()\n    if output.exists() and any(output.iterdir()):\n        parser.error(f"Refusing to overwrite non-empty output directory: {output}")\n    output.mkdir(parents=True, exist_ok=True)\n\n    samples = sorted(path for path in results.iterdir()\n                     if path.is_dir() and (path / "completion.png").is_file())\n    if not samples:\n        parser.error(f"No completed sample folders found under {results}")\n\n    tile = 256\n    label_h = 22\n    gap = 8\n    columns = 1\n    rows = (len(samples) + columns - 1) // columns\n    sheet = Image.new("RGB", (4 * tile + 5 * gap,\n                               rows * (tile + label_h) + (rows + 1) * gap), "#20242b")\n    draw = ImageDraw.Draw(sheet)\n    font = ImageFont.load_default()\n    report = {\n        "experiment": "GrabCut foreground estimate and visible-pixel-preserving composite",\n        "white_threshold": args.white_threshold,\n        "occlusion_ordering": "not modeled; generated pixels can cover input occluders",\n        "samples": [],\n    }\n\n    for index, sample in enumerate(samples):\n        source = np.asarray(Image.open(sample / "input.png").convert("RGB").resize((tile, tile)))\n        generated = np.asarray(Image.open(sample / "completion.png").convert("RGB").resize((tile, tile)))\n        visible = np.asarray(Image.open(sample / "visible_mask.png").convert("L").resize(\n            (tile, tile), Image.Resampling.NEAREST)) >= 128\n        predicted_foreground = estimate_foreground(generated, visible, args.white_threshold)\n        fill_hidden = predicted_foreground & ~visible\n        composite = source.copy()\n        composite[fill_hidden] = generated[fill_hidden]\n        mask_image = np.where(predicted_foreground, 255, 0).astype(np.uint8)\n\n        sample_out = output / sample.name\n        sample_out.mkdir(parents=True, exist_ok=True)\n        Image.fromarray(mask_image).save(sample_out / "estimated_foreground.png")\n        Image.fromarray(composite).save(sample_out / "composite_preview.png")\n\n        panels = [Image.fromarray(source), Image.fromarray(generated),\n                  Image.fromarray(mask_image).convert("RGB"), Image.fromarray(composite)]\n        labels = ["Input", "Raw completion", "Estimated foreground", "Composite preview"]\n        row, col = divmod(index, columns)\n        for j, (panel, label) in enumerate(zip(panels, labels)):\n            x = gap + j * (tile + gap)\n            y = gap + row * (tile + label_h + gap)\n            if index == 0:\n                draw.text((x, y), label, fill="#d8dee9", font=font)\n            sheet.paste(panel, (x, y + label_h))\n        draw.text((gap,\n                   gap + row * (tile + label_h + gap) + 2), sample.name,\n                  fill="#d8dee9", font=font)\n        report["samples"].append({\n            "sample_id": sample.name,\n            "foreground_fraction": round(float(predicted_foreground.mean()), 4),\n            "added_pixel_fraction": round(float(fill_hidden.mean()), 4),\n            "composite": f"{sample.name}/composite_preview.png",\n            "estimated_foreground": f"{sample.name}/estimated_foreground.png",\n        })\n\n    sheet.save(output / "contact_sheet.png")\n    (output / "experiment.json").write_text(json.dumps(report, indent=2) + "\\n", encoding="utf-8")\n    (output / "README.txt").write_text(\n        "Experimental previews only. Foreground is estimated with GrabCut using "\n        "border-connected near-white pixels as background and the visible-object "\n        "mask as foreground hints. It may still miss white object details or retain "\n        "gray artifacts. Original pixels under the visible-object mask are preserved. "\n        "The preview does not model whether hidden object pixels belong behind an "\n        "occluder, so generated pixels may paint over the occluding object. Inspect "\n        "before using; this is not a validated final output.\\n", encoding="utf-8")\n    print(json.dumps({"output": str(output), "samples": len(samples),\n                      "contact_sheet": str(output / "contact_sheet.png")}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/compose_scene_completion.py': '"""Place a Pix2Gestalt object completion back into its original full-scene image.\n\nThis is a scene-level visualization/compositing step, not a scene generator. It\nkeeps all source pixels except the predicted hidden-object region inside the\nuser-supplied occluder mask. The completed object is also exported as a full-frame\ntransparent layer so downstream tools can choose a different depth ordering.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nfrom compose_rgb_results import estimate_foreground\n\n\ndef load_model_mask(path: Path, scene_size: tuple[int, int], crop: tuple[int, int, int, int],\n                    model_size: tuple[int, int]) -> np.ndarray:\n    """Load a mask defined either in scene coordinates or completion-crop coordinates."""\n    mask_image = Image.open(path).convert("L")\n    if mask_image.size == scene_size:\n        mask_image = mask_image.crop(crop)\n    return np.asarray(mask_image.resize(model_size, Image.Resampling.NEAREST)) >= 128\n\n\ndef load_crop_mask(path: Path, scene_size: tuple[int, int], crop: tuple[int, int, int, int],\n                   crop_size: tuple[int, int]) -> np.ndarray:\n    """Load a mask and align it with the original-resolution scene crop."""\n    mask_image = Image.open(path).convert("L")\n    if mask_image.size == scene_size:\n        mask_image = mask_image.crop(crop)\n    return np.asarray(mask_image.resize(crop_size, Image.Resampling.NEAREST)) >= 128\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--run-dir", type=Path,\n                        help="Pix2Gestalt run folder created with --scene and --crop.")\n    parser.add_argument("--scene", type=Path,\n                        help="Full, unmodified scene photo used to prepare the object crop.")\n    parser.add_argument("--completion", type=Path,\n                        help="Pix2Gestalt RGB completion for one object crop.")\n    parser.add_argument("--visible-mask", type=Path,\n                        help="Visible target-object mask; may be full-scene or crop-sized.")\n    parser.add_argument("--occluder-mask", type=Path,\n                        help="Mask of the foreground obstruction; may be full-scene or crop-sized.")\n    parser.add_argument("--completion-mask", type=Path,\n                        help="Optional predicted full-object mask aligned with completion RGB.")\n    parser.add_argument("--crop", nargs=4, type=int, metavar=("X0", "Y0", "X1", "Y1"),\n                        help="Original-scene crop used for inference, in pixel coordinates.")\n    parser.add_argument("--output-dir", required=True, type=Path)\n    parser.add_argument("--white-threshold", type=int, default=12,\n                        help="GrabCut border-background tolerance for white-backed RGB outputs.")\n    args = parser.parse_args()\n\n    if not 0 <= args.white_threshold <= 254:\n        parser.error("--white-threshold must be in [0, 254]")\n    if args.run_dir:\n        if any((args.scene, args.completion, args.visible_mask, args.crop)):\n            parser.error("Use --run-dir or the explicit --scene/--completion/--visible-mask/--crop inputs, not both.")\n        run_dir = args.run_dir.resolve()\n        metadata_path = run_dir / "scene_metadata.json"\n        if not metadata_path.is_file():\n            parser.error(f"{run_dir} has no scene_metadata.json; rerun Pix2Gestalt with --scene and --crop.")\n        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))\n        args.scene = run_dir / "scene.png"\n        args.completion = run_dir / "completion.png"\n        args.visible_mask = run_dir / "visible_mask.png"\n        args.crop = metadata["crop_xyxy"]\n        if args.occluder_mask is None:\n            saved_occluder = run_dir / "occluder_mask.png"\n            if saved_occluder.is_file():\n                args.occluder_mask = saved_occluder\n        for required in (args.scene, args.completion, args.visible_mask):\n            if not required.is_file():\n                parser.error(f"Missing run artifact: {required}")\n    elif not all((args.scene, args.completion, args.visible_mask, args.crop)):\n        parser.error("Provide --run-dir or all of --scene, --completion, --visible-mask, and --crop.")\n    if args.occluder_mask is None:\n        parser.error("Provide --occluder-mask or save it into the Pix2Gestalt run with --occluder-mask.")\n\n    args.scene = args.scene.resolve()\n    args.completion = args.completion.resolve()\n    args.visible_mask = args.visible_mask.resolve()\n    args.occluder_mask = args.occluder_mask.resolve()\n    if args.completion_mask:\n        args.completion_mask = args.completion_mask.resolve()\n    scene_image = Image.open(args.scene).convert("RGB")\n    scene_size = scene_image.size\n    x0, y0, x1, y1 = args.crop\n    if not (0 <= x0 < x1 <= scene_size[0] and 0 <= y0 < y1 <= scene_size[1]):\n        parser.error(f"--crop {args.crop} is outside scene dimensions {scene_size}")\n    crop = (x0, y0, x1, y1)\n    crop_size = (x1 - x0, y1 - y0)\n\n    output = args.output_dir.resolve()\n    if output.exists() and any(output.iterdir()):\n        parser.error(f"Refusing to overwrite non-empty output directory: {output}")\n    output.mkdir(parents=True, exist_ok=True)\n\n    completion_image = Image.open(args.completion).convert("RGBA")\n    model_size = completion_image.size\n    completion_rgba = np.asarray(completion_image)\n    completion_rgb = completion_rgba[..., :3]\n    visible_model = load_model_mask(args.visible_mask, scene_size, crop, model_size)\n    if not visible_model.any():\n        parser.error("The visible-object mask is empty inside the inference crop.")\n\n    if args.completion_mask:\n        predicted_model = load_model_mask(args.completion_mask, scene_size, crop, model_size)\n        mask_method = "provided completion mask"\n    elif np.any(completion_rgba[..., 3] < 250):\n        predicted_model = completion_rgba[..., 3] >= 128\n        mask_method = "completion PNG alpha channel"\n    else:\n        predicted_model = estimate_foreground(completion_rgb, visible_model, args.white_threshold)\n        mask_method = "GrabCut seeded by visible-object mask and near-white border"\n\n    # Preserve observed target support even if foreground extraction drops pale details.\n    predicted_model |= visible_model\n    scene_crop = scene_image.crop(crop)\n    generated_crop = completion_image.convert("RGB").resize(crop_size, Image.Resampling.LANCZOS)\n    generated_rgb = np.asarray(generated_crop)\n    visible_crop = load_crop_mask(args.visible_mask, scene_size, crop, crop_size)\n    predicted_crop = np.asarray(\n        Image.fromarray(predicted_model.astype(np.uint8) * 255).resize(\n            crop_size, Image.Resampling.NEAREST)\n    ) >= 128\n    occluder_crop = load_crop_mask(args.occluder_mask, scene_size, crop, crop_size)\n    hidden_crop = predicted_crop & ~visible_crop\n    reveal_crop = hidden_crop & occluder_crop\n\n    # Full-frame transparent layer: source RGB for observed pixels, generated RGB\n    # for hidden pixels. Alpha lets a downstream renderer place it at any depth.\n    object_rgb = generated_rgb.copy()\n    source_crop_rgb = np.asarray(scene_crop)\n    object_rgb[visible_crop] = source_crop_rgb[visible_crop]\n    object_alpha = predicted_crop.astype(np.uint8) * 255\n    object_layer = np.dstack([object_rgb, object_alpha])\n    full_layer = Image.new("RGBA", scene_size, (0, 0, 0, 0))\n    full_layer.paste(Image.fromarray(object_layer, mode="RGBA"), (x0, y0))\n    full_layer.save(output / "completed_object_layer.png")\n\n    # Reveal only the predicted hidden portion that overlaps the marked obstruction.\n    # This deliberately replaces those foreground pixels to make the completion\n    # visible; it does not claim to preserve physically correct occlusion ordering.\n    scene_result = np.asarray(scene_image).copy()\n    scene_result[y0:y1, x0:x1][reveal_crop] = generated_rgb[reveal_crop]\n    Image.fromarray(scene_result).save(output / "scene_completion.png")\n\n    diagnostic = np.zeros((scene_size[1], scene_size[0], 3), dtype=np.uint8)\n    diagnostic[y0:y1, x0:x1][occluder_crop] = (45, 100, 255)\n    diagnostic[y0:y1, x0:x1][visible_crop] = (255, 175, 30)\n    diagnostic[y0:y1, x0:x1][reveal_crop] = (45, 220, 100)\n    Image.fromarray(diagnostic).save(output / "scene_masks_preview.png")\n\n    report = {\n        "scene": str(args.scene.resolve()),\n        "completion": str(args.completion.resolve()),\n        "crop_xyxy": list(crop),\n        "scene_size": list(scene_size),\n        "completion_size": list(model_size),\n        "mask_method": mask_method,\n        "predicted_hidden_pixels": int(hidden_crop.sum()),\n        "revealed_pixels_inside_occluder": int(reveal_crop.sum()),\n        "output_images": {\n            "scene_completion.png": "full-scene image with predicted hidden target pixels revealed over the marked occluder",\n            "completed_object_layer.png": "full-frame transparent object layer for downstream depth compositing",\n            "scene_masks_preview.png": "blue=occluder, orange=observed object, green=predicted pixels revealed",\n        },\n        "limitations": [\n            "This composites an object-level Pix2Gestalt result into the original scene; it is not a full-scene generative model.",\n            "The reveal preview replaces marked occluder pixels where the predicted hidden object overlaps them; this is for visualization and does not preserve physical depth ordering.",\n            "Foreground extraction can misclassify white object details; pass --completion-mask for a supplied amodal mask when available.",\n            "The crop coordinates must match the exact scene crop used during model inference.",\n            "Generated hidden appearance is a plausible prediction, not verified recovery of the unseen pixels.",\n        ],\n    }\n    (output / "summary.json").write_text(json.dumps(report, indent=2) + "\\n", encoding="utf-8")\n    print(json.dumps({"output_dir": str(output), **report["output_images"],\n                      "revealed_pixels": int(reveal_crop.sum())}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/compose_scene_batch.py': '"""Compose several object completions into one full-scene preview.\n\nEach object\'s crop completion is processed by compose_scene_completion.py. The\nresult is then used as the base for the next object, preserving earlier reveals.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\n\ndef resolve(manifest_dir: Path, value: str) -> Path:\n    path = Path(value)\n    return path if path.is_absolute() else (manifest_dir / path).resolve()\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--scene", required=True, type=Path, help="Original full-scene photo.")\n    parser.add_argument("--manifest", required=True, type=Path,\n                        help="JSON file listing one or more Pix2Gestalt object completions.")\n    parser.add_argument("--output-dir", required=True, type=Path)\n    args = parser.parse_args()\n\n    scene = args.scene.resolve()\n    manifest_path = args.manifest.resolve()\n    manifest_dir = manifest_path.parent\n    data = json.loads(manifest_path.read_text(encoding="utf-8"))\n    objects = data.get("objects") if isinstance(data, dict) else None\n    if not isinstance(objects, list) or not objects:\n        parser.error("Manifest must contain a non-empty \'objects\' list.")\n\n    output = args.output_dir.resolve()\n    if output.exists() and any(output.iterdir()):\n        parser.error(f"Refusing to overwrite non-empty output directory: {output}")\n    output.mkdir(parents=True, exist_ok=True)\n    object_outputs = output / "objects"\n    object_outputs.mkdir()\n    compositor = Path(__file__).with_name("compose_scene_completion.py")\n    current_scene = scene\n    records = []\n\n    for index, item in enumerate(objects, start=1):\n        name = str(item.get("name", f"object-{index:02d}"))\n        if not name or Path(name).name != name:\n            parser.error(f"Invalid object name: {name!r}")\n        required = ("completion", "visible_mask", "occluder_mask", "crop")\n        if any(key not in item for key in required):\n            parser.error(f"Object {name!r} must define {\', \'.join(required)}.")\n        crop = item["crop"]\n        if not isinstance(crop, list) or len(crop) != 4:\n            parser.error(f"Object {name!r} crop must be [x0, y0, x1, y1].")\n\n        sample_output = object_outputs / name\n        command = [\n            sys.executable, str(compositor),\n            "--scene", str(current_scene),\n            "--completion", str(resolve(manifest_dir, item["completion"])),\n            "--visible-mask", str(resolve(manifest_dir, item["visible_mask"])),\n            "--occluder-mask", str(resolve(manifest_dir, item["occluder_mask"])),\n            "--crop", *map(str, crop),\n            "--output-dir", str(sample_output),\n        ]\n        if item.get("completion_mask"):\n            command.extend(["--completion-mask", str(resolve(manifest_dir, item["completion_mask"]))])\n        subprocess.run(command, check=True)\n        current_scene = sample_output / "scene_completion.png"\n        records.append({"name": name, "output": str(current_scene),\n                        "crop_xyxy": crop, "layer": str(sample_output / "completed_object_layer.png")})\n\n    final_scene = output / "scene_completion.png"\n    shutil.copy2(current_scene, final_scene)\n    summary = {\n        "scene": str(scene),\n        "objects_composed_in_order": records,\n        "final_scene": str(final_scene),\n        "composition": "Each completion reveals predicted hidden pixels inside its marked occluder overlap; later objects are composed over earlier results.",\n        "limitation": "This is a full-frame visualization built from object-level predictions, not a depth-aware scene-generation model.",\n    }\n    (output / "summary.json").write_text(json.dumps(summary, indent=2) + "\\n", encoding="utf-8")\n    print(json.dumps({"output_dir": str(output), "scene_completion": str(final_scene),\n                      "objects_composed": len(records)}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, contents in FILES.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents)
subprocess.run([sys.executable, str(ROOT / 'scripts/setup_colab.py')], check=True)
PYTHON = ROOT / '.venv/bin/python'

## 2. Download the authors' pretrained checkpoint
Source: https://huggingface.co/cvlab/pix2gestalt-weights

The notebook uses the authors' five-epoch checkpoint for a research baseline.
The weights are CC BY-NC 4.0. This is pretrained capability, not our trained contribution.
The download resumes when the server supports byte ranges.


In [ ]:
import requests
from tqdm.auto import tqdm
checkpoint=ROOT/'checkpoints/epoch=000005.ckpt'
checkpoint.parent.mkdir(exist_ok=True)
partial=checkpoint.with_suffix('.ckpt.part')
url='https://huggingface.co/cvlab/pix2gestalt-weights/resolve/main/epoch%3D000005.ckpt'
if not checkpoint.exists():
    start=partial.stat().st_size if partial.exists() else 0
    response=requests.get(url,headers={'Range':f'bytes={start}-'} if start else {},
                          stream=True,timeout=(30,120))
    response.raise_for_status()
    resumed=response.status_code==206
    if not resumed: start=0
    total=start+int(response.headers.get('content-length',0))
    with partial.open('ab' if resumed else 'wb') as stream:
        with tqdm(total=total or None,initial=start,unit='B',unit_scale=True) as bar:
            for chunk in response.iter_content(8*1024*1024):
                if chunk:
                    stream.write(chunk)
                    bar.update(len(chunk))
    if total and partial.stat().st_size!=total:
        raise RuntimeError('Download incomplete. Rerun this cell to resume.')
    partial.rename(checkpoint)
print('Checkpoint size GB:',round(checkpoint.stat().st_size/1e9,2))


## 3. Upload Pix2Gestalt inputs

Upload one ZIP containing `image.png` (the target-object crop) and `visible_mask.png` (same crop; white marks visible target pixels). To create a full-scene result, also include the untouched `scene.png`, `occluder_mask.png`, and `crop.json` with `{"crop_xyxy": [x0, y0, x1, y1]}` coordinates in the original scene. The obstruction mask may match either the full scene or the model crop.


In [ ]:
from google.colab import files
from pathlib import Path
from PIL import Image
from IPython.display import display
import zipfile
import io
import json
uploaded=files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one ZIP of Pix2Gestalt inputs")
inputs=ROOT/"inputs"/"scene-test"
inputs.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    names=set(archive.namelist())
    required={"image.png","visible_mask.png"}
    if not required.issubset(names):
        raise ValueError("ZIP must contain image.png and visible_mask.png")
    for name in ("image.png","visible_mask.png","scene.png","occluder_mask.png","crop.json"):
        if name in names:
            (inputs/name).write_bytes(archive.read(name))
if (inputs/"occluder_mask.png").exists() and not (inputs/"scene.png").exists():
    raise ValueError("occluder_mask.png requires scene.png and crop.json for scene composition")
if (inputs/"scene.png").exists() != (inputs/"crop.json").exists():
    raise ValueError("scene.png and crop.json must be supplied together")
print("Pix2Gestalt crop and visible mask:")
display(Image.open(inputs/"image.png"), Image.open(inputs/"visible_mask.png"))
if (inputs/"scene.png").exists():
    crop_info=json.loads((inputs/"crop.json").read_text())
    print("Full scene and crop coordinates:", crop_info["crop_xyxy"])
    display(Image.open(inputs/"scene.png"))
if (inputs/"occluder_mask.png").exists():
    print("Marked obstruction mask:")
    display(Image.open(inputs/"occluder_mask.png"))


## 4. Run one completion
Initial settings: **256 × 256**, **one candidate**, **50 DDIM steps**, and automatic
mixed precision. The experimental loader maps checkpoint tensors from disk and loads
EMA denoiser weights without retaining an extra EMA model on the GPU.

These memory changes have not been validated on this free runtime. If it fails,
retain the traceback: the first milestone is not passed until an output is produced.


In [ ]:
import json
import subprocess
from IPython.display import Image as IPyImage, display
command=[str(PYTHON),str(ROOT/"scripts/run_baseline.py"),
         "--image",str(inputs/"image.png"),
         "--mask",str(inputs/"visible_mask.png"),
         "--checkpoint",str(checkpoint),"--steps","50","--seed","42"]
scene_ready=(inputs/"scene.png").exists() and (inputs/"crop.json").exists()
if scene_ready:
    crop_info=json.loads((inputs/"crop.json").read_text())
    command += ["--scene",str(inputs/"scene.png"),"--crop",*map(str,crop_info["crop_xyxy"])]
    if (inputs/"occluder_mask.png").exists():
        command += ["--occluder-mask",str(inputs/"occluder_mask.png")]
subprocess.run(command,check=True)
run_dirs=[p for p in (ROOT/"runs").iterdir() if p.is_dir() and (p/"completion.png").is_file()]
run_dir=max(run_dirs,key=lambda p:p.stat().st_mtime)
print("Pix2Gestalt run:",run_dir)
if (run_dir/"occluder_mask.png").exists():
    scene_output=run_dir/"scene-completion"
    subprocess.run([str(PYTHON),str(ROOT/"scripts/compose_scene_completion.py"),
                    "--run-dir",str(run_dir),"--output-dir",str(scene_output)],check=True)
    print("Full-scene completion preview:")
    display(IPyImage(filename=str(scene_output/"scene_completion.png")))
    print("Mask alignment review:")
    display(IPyImage(filename=str(scene_output/"scene_masks_preview.png")))
else:
    print("Object completion only. Add scene.png, crop.json, and occluder_mask.png to the ZIP for a full-scene preview.")
    display(IPyImage(filename=str(run_dir/"completion.png")))


## 5. Inspect and save the result
Check whether the output completes the selected object, preserves visible details,
and produces a plausible silhouette. The first result is an RGB completion;
clean transparent-object export is a later milestone.


In [ ]:
## What counts as this baseline milestone?

- Pix2Gestalt generated full-color completed-object images from a photo and visible-object mask.
- The fixed 12-example RGB batch completed in the saved Colab notebook: 12 inference runs succeeded, with per-run settings and resource records.
- The next main-project step is to retrieve and visually review those RGB results, then choose the next RGB-completion experiment based on observed failures.

The compact U-Net is a separate silhouette-mask experiment. Its mask predictions are not the RGB completion output.

## What counts as milestone 1 complete?
- A completion PNG was produced from the uploaded image and visible mask.
- The run includes seed, settings, upstream revisions, timing, and peak memory.
- We inspect its successes and failures before changing the model.

Next: fixed baseline examples → synthetic occlusion pairs → custom silhouette U-Net
→ experimentally test guidance → add selection and export UI.


## Fixed RGB completion evaluation

The next section runs the actual RGB generator (Pix2Gestalt) on the fixed 12-image held-out set. It does not use the silhouette U-Net. The archive contains reference masks for evaluation, but the runner passes only each photo and its visible-object mask to the generator. Inference is one image at a time; outputs include a contact sheet, per-sample records, failures, and a ZIP to download.

In [ ]:
from google.colab import files
from pathlib import Path
import subprocess
from IPython.display import Image as IPyImage, display

uploaded_eval = files.upload()  # Choose runs/mp3d-eval-v1.zip from the project folder.
archive_name = 'mp3d-eval-v1.zip'
if archive_name not in uploaded_eval:
    raise ValueError(f'Please upload the project archive named {archive_name}.')
eval_archive = ROOT/'inputs'/archive_name
eval_archive.parent.mkdir(parents=True, exist_ok=True)
eval_archive.write_bytes(uploaded_eval[archive_name])

command = [str(PYTHON), str(ROOT/'scripts/evaluate_rgb_baseline_batch.py'),
           '--root', str(ROOT), '--archive', str(eval_archive),
           '--checkpoint', str(checkpoint), '--steps', '50', '--seed', '42']
subprocess.run(command, check=True)
contact_sheet = ROOT/'runs/rgb-eval-v1/contact_sheet.png'
display(IPyImage(filename=str(contact_sheet)))
result_zip = ROOT/'runs/rgb-eval-v1.zip'
files.download(str(result_zip))